In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "source": ["# 01 - Ingest\nIngest raw data into the **Bronze** layer using Auto Loader."]
  },
  {
   "cell_type": "code",
   "source": [
    "from pyspark.sql.functions import current_timestamp\n",
    "\n",
    "catalog = \"main\"\n",
    "schema = \"bronze\"\n",
    "table_name = \"events_bronze\"\n",
    "\n",
    "raw_path = \"/mnt/raw/events/\"\n",
    "checkpoint_path = \"/mnt/checkpoints/events_bronze/\"\n",
    "\n",
    "full_table_name = f\"{catalog}.{schema}.{table_name}\"\n",
    "\n",
    "df = (\n",
    "    spark.readStream\n",
    "        .format(\"cloudFiles\")\n",
    "        .option(\"cloudFiles.format\", \"json\")\n",
    "        .option(\"cloudFiles.inferColumnTypes\", \"true\")\n",
    "        .load(raw_path)\n",
    "        .withColumn(\"ingest_timestamp\", current_timestamp())\n",
    ")\n",
    "\n",
    "(\n",
    "    df.writeStream\n",
    "      .format(\"delta\")\n",
    "      .option(\"checkpointLocation\", checkpoint_path)\n",
    "      .outputMode(\"append\")\n",
    "      .trigger(availableNow=True)\n",
    "      .toTable(full_table_name)\n",
    ")\n",
    "\n",
    "display(df)"
   ]
  }
 ],
 "metadata": {},
 "nbformat": 4,
 "nbformat_minor": 5
}